# 15 — Bias & fairness

**Purpose:** Single place for all bias + fairness evidence. Addresses AM2 criteria:

- **K20** — sources of error + algorithmic bias
- **K26** — ethics in ML use
- Per-source / per-nation / per-class / temporal fairness slices

This consolidates: nb 09 SHAP bias analysis, `src/inference/fairness_audit.py`, the Four Nations gap surfaced in curator feedback `project-curator-feedback-2026-05-05`, and the curator-override-rate framework.


In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import precision_recall_fscore_support, classification_report
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = Path("..").resolve()
load_dotenv(ROOT / ".env")
sys.path.insert(0, str(ROOT))

MODEL_DIR = ROOT / "models" / "runs" / "v1_2026-05-16"
with open(MODEL_DIR / "run_metadata.json") as f:
    run_meta = json.load(f)

VAL_CSV = ROOT / "data" / "modelling" / "val.csv"
val_df = pd.read_csv(VAL_CSV)
print(f"Val set: {len(val_df)} items")


## Part 1 — Algorithmic bias (SHAP recap)

The strongest bias finding from this project: **SBERT + metadata classifier had 27.6% feature attribution on `item_type` and `org_broad_category` rather than article content**. The model was classifying by source-type proxy, not topic.

Decision: ship the lower-F1 no-meta model (0.750 vs 0.765) for content-faithful classification. See `notebooks/09_shap_sst.ipynb` and `project-am2-deadline` memory.

Below: re-render the key SHAP figure for AM2 portfolio convenience.

In [ ]:
# Display saved SHAP plots from outputs/shap_plots/
shap_dir = ROOT / 'outputs' / 'shap_plots'
plot_files = sorted(shap_dir.rglob('*.png'))
print(f'SHAP plots available: {len(plot_files)}')
for p in plot_files[:5]:
    print(f'  {p.relative_to(ROOT)}')

# Pick the most-relevant 'global' SHAP plot.
global_plots = list((shap_dir / 'global').glob('*.png')) if (shap_dir / 'global').exists() else []
if global_plots:
    from IPython.display import Image
    display(Image(filename=str(global_plots[0])))


## Part 2 — Per-class fairness (current model on val)

In [ ]:
# Run the existing fairness audit logic in-notebook.
encoder = SentenceTransformer(run_meta['embedding_model'])
classifier = joblib.load(MODEL_DIR / "classifier.joblib")

title_col = 'title' if 'title' in val_df.columns else 'text'
target_col = 'target' if 'target' in val_df.columns else 'theme'

X = encoder.encode(val_df[title_col].fillna('').astype(str).tolist(), show_progress_bar=True)
y_true = val_df[target_col].astype(str).values
y_pred = classifier.predict(X)

# Per-class precision/recall/F1/support.
prec, rec, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=classifier.classes_, zero_division=0)
fairness_df = pd.DataFrame({
    'class': classifier.classes_,
    'precision': prec,
    'recall': rec,
    'f1': f1,
    'support': support,
}).sort_values('f1', ascending=False)
print(fairness_df.to_string(index=False))


In [ ]:
# Visual: per-class F1 bar chart, highlight under-supported classes.
fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#e74c3c' if s < 25 else '#3498db' for s in fairness_df['support']]
ax.bar(fairness_df['class'], fairness_df['f1'], color=colors)
ax.axhline(fairness_df['f1'].mean(), color='grey', linestyle='--', alpha=0.5, label=f"Mean F1 ({fairness_df['f1'].mean():.3f})")
ax.set_xticklabels(fairness_df['class'], rotation=20, ha='right')
ax.set_ylabel('F1 score')
ax.set_title('Per-class F1 — red bars indicate low support (n<25)')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'fairness_per_class.png', dpi=120)
plt.show()


## Part 3 — Per-source fairness

Does the model perform equally well across sources? If Schools Week is well represented and Belfast Telegraph isn't, the model may systematically mis-serve one of them.

This requires a source label per article in val.csv — adjust if your column is named differently.

In [ ]:
# Adjust the column name if needed.
src_col_candidates = ['source', 'organisation', 'org_broad_category']
src_col = next((c for c in src_col_candidates if c in val_df.columns), None)
if not src_col:
    print(f"No source column found in val.csv — skipping per-source analysis.")
else:
    val_df['_y_true'] = y_true
    val_df['_y_pred'] = y_pred
    val_df['_correct'] = val_df['_y_true'] == val_df['_y_pred']

    by_source = val_df.groupby(src_col).agg(
        n=('_correct', 'size'),
        acc=('_correct', 'mean'),
    ).reset_index()
    by_source = by_source[by_source['n'] >= 5].sort_values('acc', ascending=False)
    print(by_source.to_string(index=False))

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.barh(by_source[src_col], by_source['acc'], color='#3498db')
    ax.axvline(by_source['acc'].mean(), color='grey', linestyle='--')
    ax.set_xlabel('Accuracy')
    ax.set_title(f'Per-source accuracy ({src_col}, n≥5)')
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / 'fairness_per_source.png', dpi=120)
    plt.show()


## Part 4 — Per-nation fairness (Four Nations gap)

Curator (Gemma, 5 May meeting) flagged that Four Nations articles are only 0.4% of pulled articles. Test: does the model perform equally well on England vs Scotland/Wales/NI content?

Uses the new `geographic_focus` column from migration 013 (Phase 5b enrichment) — pulled from Supabase since it's not in val.csv.

In [ ]:
# Pull geographic_focus for val articles from Supabase.
import os
try:
    from supabase import create_client
    client = create_client(os.environ["SUPABASE_URL"],
                           os.environ.get("SUPABASE_ANON_KEY") or os.environ["SUPABASE_SERVICE_KEY"])
    # Match by URL (val.csv has 'link', articles table has 'url')
    url_col = 'link' if 'link' in val_df.columns else 'url'
    val_urls = val_df[url_col].dropna().tolist()
    # Supabase IN clause limit — batch if needed.
    BATCH = 100
    geo_data = []
    for i in range(0, len(val_urls), BATCH):
        chunk = val_urls[i:i+BATCH]
        r = client.table('articles').select('url, geographic_focus').in_('url', chunk).execute()
        geo_data.extend(r.data or [])
    geo_df = pd.DataFrame(geo_data)
    print(f"Geo-tagged val articles: {len(geo_df)} of {len(val_df)}")

    val_df_geo = val_df.merge(geo_df, left_on=url_col, right_on='url', how='left')
    val_df_geo['_correct'] = val_df_geo['_y_true'] == val_df_geo['_y_pred']
    by_nation = val_df_geo.dropna(subset=['geographic_focus']).groupby('geographic_focus').agg(
        n=('_correct', 'size'),
        acc=('_correct', 'mean'),
    ).reset_index().sort_values('n', ascending=False)
    print(by_nation.to_string(index=False))
except Exception as e:
    print(f"Could not fetch geographic_focus from Supabase: {e}")
    print("Skipping per-nation fairness — re-run when Supabase connection available.")


## Part 5 — Fairness over time

Does per-class fairness stay stable across the 18-week drift window? Drift in **fairness** is more concerning than drift in overall F1.

In [ ]:
# Per-class F1 over time using the weekly classified CSVs.
WEEKLY_DIR = ROOT / "data" / "modelling" / "weekly"
import re

weekly_frames = []
for f in sorted(WEEKLY_DIR.glob("classified_week_*.csv")):
    m = re.search(r"week_(\d+)", f.name)
    if not m:
        continue
    df = pd.read_csv(f)
    df['_week'] = int(m.group(1))
    weekly_frames.append(df)

if weekly_frames:
    all_weeks = pd.concat(weekly_frames, ignore_index=True)
    # If the weekly CSVs have curator decisions, we can compute per-class
    # precision over time. Otherwise this is a confidence-stability check.
    if 'curator_label' in all_weeks.columns and 'top1' in all_weeks.columns:
        # Real per-class precision per week.
        results = []
        for week, g in all_weeks.dropna(subset=['curator_label']).groupby('_week'):
            for cls in g['curator_label'].unique():
                cls_pred = g[g['top1'] == cls]
                if len(cls_pred) >= 3:
                    prec = (cls_pred['curator_label'] == cls).mean()
                    results.append({'week': week, 'class': cls, 'precision': prec, 'n': len(cls_pred)})
        per_week_class = pd.DataFrame(results)
        print(per_week_class.to_string(index=False))
    else:
        # Fall back to confidence-by-class-by-week.
        per_week_class = (all_weeks.groupby(['_week', 'top1'])['top1_confidence'].mean().reset_index())
        print(per_week_class.head(20).to_string(index=False))
else:
    print("No weekly CSVs found.")


## Part 6 — Curator-override-rate fairness signal

When the curator overrides the model (picks Top 2 or manual), that's a fairness signal: the model is systematically wrong for certain classes/sources. Pull from `curator_decisions`.

In [ ]:
# Read curator_decisions. Each row has action ∈ {accept_top1, accept_top2,
# manual, reject, keep, summary_only}. Override rate = (accept_top2 + manual) /
# (accept_top1 + accept_top2 + manual).
try:
    r = client.table('curator_decisions').select('url, action, label').execute()
    decisions = pd.DataFrame(r.data or [])
    if len(decisions) == 0:
        print("No curator decisions yet (table was wiped 2026-05-26). Framework in place; data accumulates over time.")
    else:
        # Join with articles for class label.
        urls = decisions['url'].tolist()
        articles_data = []
        for i in range(0, len(urls), 100):
            chunk = urls[i:i+100]
            r2 = client.table('articles').select('url, top1').in_('url', chunk).execute()
            articles_data.extend(r2.data or [])
        article_df = pd.DataFrame(articles_data)
        merged = decisions.merge(article_df, on='url', how='left')

        # Compute override rate per class.
        merged['_kept'] = merged['action'].isin(['accept_top1', 'accept_top2', 'manual'])
        merged['_overridden'] = merged['action'].isin(['accept_top2', 'manual'])

        kept = merged[merged['_kept']]
        override_rate = kept.groupby('top1')['_overridden'].agg(['sum', 'count', 'mean']).reset_index()
        override_rate.columns = ['model_top1', 'n_overridden', 'n_kept', 'override_rate']
        print(override_rate.sort_values('override_rate', ascending=False).to_string(index=False))
except Exception as e:
    print(f"Could not pull curator_decisions: {e}")


## Conclusions

This notebook gathers the bias + fairness evidence in one place. Key threads:

1. **Algorithmic bias** (K20): SHAP-driven decision to ship no-meta model. Source-proxy classification avoided.
2. **Per-class fairness** (K20): some classes have lower support and weaker F1 — flagged honestly.
3. **Per-source fairness**: variation across sources — if Schools Week >> Belfast Telegraph, that's worth declaring.
4. **Per-nation fairness** (Four Nations): direct response to curator feedback (Gemma, 5 May).
5. **Temporal fairness**: drift in fairness, not just F1.
6. **Curator-override rate**: framework in place even if data sparse post-fresh-start.

For AM2 (K26 ethics): no mitigation is perfect, but the design choices recorded here — no metadata, curator-in-loop, transparency via FastAPI `/docs`, honest reporting of disparities — together address the criterion.